# Script2 : Approche Deep Learning (Embeddings Sémantiques Denses)

In [1]:
import os

# 1. Masquer les avertissements système PyTorch et Symlinks Windows
os.environ["TORCH_CPP_LOG_LEVEL"] = "ERROR"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# 2. Désactiver les barres de progression graphiques (évite le plantage LookupError dans Jupyter)
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TQDM_DISABLE"] = "1"

# 3. Charger ensuite vos bibliothèques
import torch
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer, util

print("Configuration effectuée avec succès !")

W1110 11:17:24.116000 15592 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Configuration effectuée avec succès !


In [2]:
# ÉTAPE 1 : Chargement des données et Initialisation du Modèle Neuronal
import re
import html

print("--> Chargement du modèle de langue neuronal SentenceTransformer...")

MODEL_NAME = "all-MiniLM-L6-v2"
model = SentenceTransformer(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print(f"Modèle chargé sur le périphérique : {device}")

# Chargement des jeux de données
corpus_df = pd.read_csv("medrxiv2026_corpus.csv")
queries_df = pd.read_csv("medrxiv2026_reco_queries_train.csv")
qrels_df = pd.read_csv("medrxiv2026_reco_qrels_train.csv")

def nettoyer_texte(texte):
    if not isinstance(texte, str):
        return ""
    texte = html.unescape(texte).lower()
    texte = re.sub(r'</?(?:p|b|i|u|sub|sup|br|em|strong|span|div|ul|ol|li)\b[^>]*>', ' ', texte)
    texte = re.sub(r'[^a-z0-9\s]', ' ', texte)
    return re.sub(r'\s+', ' ', texte).strip()

corpus_df['clean_title'] = corpus_df['title'].apply(nettoyer_texte)
corpus_df['clean_abstract'] = corpus_df['abstract'].apply(nettoyer_texte)
corpus_df['full_text'] = corpus_df['clean_title'] + ". " + corpus_df['clean_abstract']
queries_df['clean_query'] = queries_df['query'].apply(nettoyer_texte)

doc_ids = corpus_df['ID'].values

--> Chargement du modèle de langue neuronal SentenceTransformer...


Modèle chargé sur le périphérique : cpu


In [3]:
# ÉTAPE 2 : Encodage Vectoriel Dense (Calcul des Embeddings)
# ------------------------------------------------------------------------------
print("--> Calcul des embeddings denses pour l'ensemble du corpus (5058 articles)...")
# Chaque article est transformé en un vecteur continu de 384 dimensions
corpus_embeddings = model.encode(
    corpus_df['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    batch_size=32
)

print("--> Calcul des embeddings denses pour les requêtes...")
query_embeddings = model.encode(
    queries_df['query'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=False,
    batch_size=32
)

print(f"Dimension des embeddings corpus : {corpus_embeddings.shape}")
print(f"Dimension des embeddings requêtes : {query_embeddings.shape}")

--> Calcul des embeddings denses pour l'ensemble du corpus (5058 articles)...


Batches:   0%|          | 0/159 [00:00<?, ?it/s]

--> Calcul des embeddings denses pour les requêtes...
Dimension des embeddings corpus : torch.Size([5058, 384])
Dimension des embeddings requêtes : torch.Size([465, 384])


In [4]:
# ÉTAPE 3 : Métriques d'évaluation corrigées (nDCG@10 et MRR)
TOP_K = 10

def dcg_at_k(relevances, k=10):
    relevances = np.asarray(relevances, dtype=float)[:k]
    if not relevances.size:
        return 0.
    return np.sum((2**relevances - 1) / np.log2(np.arange(2, relevances.size + 2)))

def ndcg_at_k(relevances, relevances_ideales, k=10):
    """Calcule le nDCG en comparant aux vrais documents pertinents du qrels."""
    idcg = dcg_at_k(sorted(relevances_ideales, reverse=True), k)
    return dcg_at_k(relevances, k) / idcg if idcg else 0.

def rr_at_k(relevances, k=10):
    for i, r in enumerate(relevances[:k]):
        if r > 0:
            return 1.0 / (i + 1)
    return 0.

# Construction de la vérité terrain
qrels_dict = {}
for q_id, d_id, rel in qrels_df[['query_id', 'ID', 'relevance']].itertuples(index=False):
    qrels_dict.setdefault(q_id, {})[d_id] = rel

# Split train (80%) / dev (20%) identique au Script 1
rng = np.random.RandomState(42)
perm = rng.permutation(len(queries_df))
n_dev = int(0.2 * len(perm))
DEV, TRAIN = np.sort(perm[:n_dev]), np.sort(perm[n_dev:])
print(f"{len(TRAIN)} requêtes pour régler, {len(DEV)} pour valider")

372 requêtes pour régler, 93 pour valider


In [5]:
# ÉTAPE 4 : Calcul de Similarité Cosinus et Évaluation
print("--> Évaluation de la recherche sémantique...")

# Matrice de scores Cosine Similarity (nb_requêtes x nb_documents)
cosine_scores = util.cos_sim(query_embeddings, corpus_embeddings).cpu().numpy()

def evaluer_dense(S, positions):
    nd, rr = [], []
    for i in positions:
        q_id = queries_df.loc[i, 'query_id']
        top = np.argsort(-S[i], kind="stable")[:TOP_K]
        rels = [qrels_dict[q_id].get(d, 0) for d in doc_ids[top]]
        nd.append(ndcg_at_k(rels, list(qrels_dict[q_id].values()), TOP_K))
        rr.append(rr_at_k(rels, TOP_K))
    return {"nDCG@10": round(float(np.mean(nd)), 4), "MRR": round(float(np.mean(rr)), 4)}

res_deep_train = evaluer_dense(cosine_scores, TRAIN)
res_deep_dev = evaluer_dense(cosine_scores, DEV)

print("\n" + "="*50)
print(f"Deep Learning ({MODEL_NAME}) réglage   :", res_deep_train)
print(f"Deep Learning ({MODEL_NAME}) validation:", res_deep_dev)
print("="*50 + "\n")

--> Évaluation de la recherche sémantique...

Deep Learning (all-MiniLM-L6-v2) réglage   : {'nDCG@10': 0.7259, 'MRR': 0.9172}
Deep Learning (all-MiniLM-L6-v2) validation: {'nDCG@10': 0.7262, 'MRR': 0.9283}



In [6]:
# ÉTAPE 5 : Génération du Fichier de Soumission Final (Test Set)

def generer_predictions_test_deep(fichier_test_queries, fichier_sortie="Groupe5_Projet3_Script2_DeepLearning.csv"):
    """
    Calcule les recommandations sémantiques pour le jeu de test officiel
    et exporte le CSV exact demandé par le sujet : (query_id, ID, rank)
    """
    test_queries_df = pd.read_csv(fichier_test_queries)
    
    # Encodage des requêtes de test
    test_query_embeddings = model.encode(
        test_queries_df['query'].tolist(),
        convert_to_tensor=True,
        show_progress_bar=False
    )

    resultats = []
    for idx, row in test_queries_df.iterrows():
        q_id = row['query_id']
        cosine_scores = util.cos_sim(test_query_embeddings[idx], corpus_embeddings)[0]
        top_results = torch.topk(cosine_scores, k=TOP_K)
        top10_indices = top_results.indices.cpu().numpy()
        
        for rank_position, doc_idx in enumerate(top10_indices, start=1):
            doc_id = corpus_df.iloc[doc_idx]['ID']
            resultats.append({
                'query_id': q_id,
                'ID': doc_id,
                'rank': rank_position
            })
            
    df_resultats = pd.DataFrame(resultats)
    df_resultats.to_csv(fichier_sortie, index=False)
    print(f"Fichier de recommandation Deep Learning généré avec succès : {fichier_sortie}")

In [7]:
# ÉTAPE 6 : Fichier de prédictions (jeu de test)
FICHIER_TEST = "medrxiv2026_reco_queries_test.csv"
FICHIER_SORTIE = "Groupe5_Projet3_Script2_.csv"

generer_predictions_test_deep(FICHIER_TEST, FICHIER_SORTIE)

Fichier de recommandation Deep Learning généré avec succès : Groupe5_Projet3_Script2_.csv


In [8]:
# ÉTAPE 7 : Vérification du format
import os

pred = pd.read_csv(FICHIER_SORTIE)
test_q = pd.read_csv(FICHIER_TEST)

assert list(pred.columns) == ['query_id', 'ID', 'rank'], "Colonnes incorrectes"
assert set(pred['query_id']) == set(test_q['query_id']), "Requêtes manquantes ou en trop"
assert (pred.groupby('query_id').size() == 10).all(), "Il faut exactement 10 lignes par requête"
assert (pred.groupby('query_id')['rank'].apply(lambda r: sorted(r) == list(range(1, 11)))).all(), "Rangs 1 à 10 incorrects"
assert not pred.duplicated(['query_id', 'ID']).any(), "Doublon (query_id, ID)"
assert pred['ID'].isin(corpus_df['ID']).all(), "ID absent du corpus"

print(f"OK : {pred['query_id'].nunique()} requêtes, {len(pred)} lignes")
print("Fichier :", os.path.abspath(FICHIER_SORTIE))
pred.head(12)

OK : 200 requêtes, 2000 lignes
Fichier : C:\Users\ELITE COMPUTER\OneDrive - etu.unistra.fr\Bureau\SciDo\projet 2 grp 5\Groupe5_Projet3_Script2_.csv


,query_id,ID,rank
0,Q0002,1543,1
1,Q0002,4588,2
2,Q0002,4330,3
3,Q0002,3952,4
4,Q0002,5012,5
5,Q0002,3710,6
6,Q0002,4435,7
7,Q0002,2150,8
8,Q0002,3735,9
9,Q0002,3518,10
